# C1.2 · Weaponizing the ingestion path

**Function C — Red Teaming and Security Research with AI → Agentic Evaluation and Red Teaming**

Builds on **[C1.1 · Platform ingestion and supply-chain risks](https://spbreed.github.io/cyber-commons/lessons/C1.1.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Data-layer payloads that exploit a parser to reach code execution during automated embedding generation, and the provenance manifest that makes such a run traceable.

**Why a security engineer needs it.** The embedding pipeline is code, and the host it runs on is usually more privileged than the one serving traffic. A record that is data to the model and an exploit to the decoder runs there, and nothing traces it unless the ingestion was manifested.

## 1 · The hook

A dataset is not passive: it is parsed, decoded and embedded, and every step is a parser with a threat model. A crafted record reaches code execution on the indexing host — which usually has more access than the box serving traffic.

> **At CyberTravels.** The pipeline is CyberTravels' RAG ingestion: vendor PDFs and images through OCR and a decoder, on the indexing host that can reach the backend APIs. A crafted invoice is the payload.

## 2 · The framework

```
   the record is data to the model, CODE to the parser

   dataset record ---> image decoder ---> embedding job (indexing host)
                           |                     |
                    crafted header         RCE, with the indexing
                    exploits the           host's access, which is
                    decoder                usually broader than serving

   control: a provenance manifest — source, parser, digest — so a run
   that executed something traces back to the record that carried it
```

Ingestion is also where code runs. A dataset is not passive: it is parsed,
decoded and embedded, and every one of those steps is a parser with a threat
model. A crafted record that exploits an image decoder or a deserialiser
achieves **remote code execution during automated embedding generation** — on
the training or indexing host, which usually has more access than the serving
one.

Weaponising the ingestion path means treating the embedding pipeline as an
attack surface, and the research output is a provenance manifest: what was
ingested, from where, parsed by what, so a payload that ran can be traced to the
record that carried it.

## 3 · The procedure, as a skill

The skill builds a manifest of what CyberTravels' RAG pipeline ingested — source, parser, and a digest per record — so an embedding run that executed something can be traced to the record that carried it.

### The skill — [`skills/research/training-data-provenance-manifest/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/training-data-provenance-manifest/SKILL.md)

```yaml
name: training-data-provenance-manifest
description: >-
  Compute what fraction of a corpus an attacker needs to poison it, and build a
  hashed manifest that can answer where a record came from and whether it
  changed. Use when reviewing training or fine-tuning data, or a RAG corpus
  nobody can attest to.
allowed-tools: Read, Grep, Glob
```

# A list of records is not provenance

Data-layer attacks need a smaller share of a corpus than people expect, so the
useful question is not "could someone poison this" but "could we tell". A record
list answers none of the four questions that matter; a manifest of content
hashes with a root answers all four, and it is cheap.

## When to use this

Any corpus that trains, fine-tunes or grounds a model — including the RAG index
somebody built from a shared drive.

## Procedure

**1 — State the poisoning rates in absolute terms.** For the corpus size you
have, print what 0.01%, 0.1% and 1% mean as a record count. The number is
usually small enough to end the argument about whether it is feasible.

**2 — Write down the four questions.** Where did this record come from, has it
changed since ingestion, what is in the corpus now, and what was in it at
training time. These are the requirements.

**3 — Compare what each artefact can answer.** A record list, a row count, a
snapshot, a hashed manifest. Only the last answers all four, and showing the
table is more persuasive than asserting it.

**4 — Build the manifest.** Content hash per record plus its source, and a root
over the whole set. The root is what makes "the corpus changed" a one-comparison
question.

**5 — Demonstrate detection.** Append records, recompute, and show both that the
root moved and which records are new. A manifest that detects change without
localising it sends you back to diffing the corpus.

## Example

**Input** — the fixture committed at the top of [`scripts/training_data_provenance_manifest.py`](scripts/training_data_provenance_manifest.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
   10 poisoned of 100,000 → 0.01000%
  100 poisoned of 100,000 → 0.10000%
 1000 poisoned of 100,000 → 1.00000%

Published attacks land in this range. 'We have more clean data' is not
a defence, because the attacker is not trying to outvote you.
setup                               Q1   Q2   Q3   Q4   
------------------------------------------------------------
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "corpus": {"records": 0, "poison_rates": [{"rate": 0.0, "records": 0}]},
  "questions": [{"question": "str", "answerable_by": ["str"]}],
  "manifest": {"records": 0, "root": "str", "per_record": [{"id": "str", "hash": "str", "source": "str"}]},
  "detection": {"appended": 0, "root_changed": true, "localised": ["str"]}
}
```

## Failure modes

- **Arguing about feasibility.** Print the record count and the argument ends.
- **A manifest with no source field.** It answers "changed", never "from where".
- **A root with no per-record hashes.** Detection without localisation.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/training-data-provenance-manifest/scripts/training_data_provenance_manifest.py
SCRIPT = "skills/research/training-data-provenance-manifest/scripts/training_data_provenance_manifest.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The manifest names each source and the parser that touched it, and the record with no verifiable origin is flagged as the one a payload would ride in on.

## Your turn

Find the host that runs your embedding jobs and check what it can reach. It is usually the most privileged machine nobody threat-modelled.

---

**Next → [C1.3 · Cognitive vulnerability and elicitation scaling](https://spbreed.github.io/cyber-commons/lessons/C1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*